In [1]:
import tensorflow as tf
tf.keras.backend.clear_session()

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import KFold

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import time
import random

In [3]:
seed = 42

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cpu


In [5]:
transform = transforms.ToTensor()

train_dataset = torchvision.datasets.KMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.KMNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

print("Training samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

Training samples: 60000
Test samples: 10000


In [6]:
test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [7]:
class KMNISTModel(nn.Module):

    def __init__(self):
        super(KMNISTModel, self).__init__()

        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

        self.relu = nn.ReLU()

    def forward(self, x):

        x = x.view(-1, 784)

        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))

        x = self.fc3(x)

        return x

In [8]:
model = KMNISTModel().to(device)

print(model)

KMNISTModel(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=10, bias=True)
  (relu): ReLU()
)


In [9]:
criterion = nn.CrossEntropyLoss()

In [10]:
def calculate_accuracy(model, loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    return accuracy

In [11]:
## Model training 

def train_model(model, train_loader, optimizer, epochs=20):

    train_losses = []
    train_accuracies = []

    start_time = time.time()

    for epoch in range(epochs):

        model.train()

        running_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100 * correct / total

        train_losses.append(epoch_loss)
        train_accuracies.append(epoch_acc)

        print(f"Epoch [{epoch+1}/{epochs}] "
              f"Loss: {epoch_loss:.4f} "
              f"Accuracy: {epoch_acc:.2f}%")

    training_time = time.time() - start_time

    return train_losses, train_accuracies, training_time

In [12]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [13]:
rmsprop_configs = [
    {"lr":0.01, "alpha":0.99},
    {"lr":0.001, "alpha":0.99},
    {"lr":0.0001, "alpha":0.95},
    {"lr":0.005, "alpha":0.90}
]

In [14]:
for config in rmsprop_configs:

    results = []
    print("\n========================")
    print("Testing RMSprop:", config)
    print("========================")

    model = KMNISTModel().to(device)

    optimizer = optim.RMSprop(
        model.parameters(),
        lr=config["lr"],
        alpha=config["alpha"]
    )

    losses, accs, train_time = train_model(
        model,
        train_loader,
        optimizer,
        epochs=10
    )

    test_acc = calculate_accuracy(
        model,
        test_loader
    )

    results.append({
        "Optimizer":"RMSprop",
        "Learning Rate":config["lr"],
        "Alpha":config["alpha"],
        "Test Accuracy":test_acc,
        "Training Time":train_time
    })

    print("Test Accuracy:", round(test_acc,2), "%")


Testing RMSprop: {'lr': 0.01, 'alpha': 0.99}
Epoch [1/10] Loss: 0.5965 Accuracy: 83.94%
Epoch [2/10] Loss: 0.3346 Accuracy: 90.21%
Epoch [3/10] Loss: 0.2862 Accuracy: 91.65%
Epoch [4/10] Loss: 0.2617 Accuracy: 92.41%
Epoch [5/10] Loss: 0.2429 Accuracy: 92.95%
Epoch [6/10] Loss: 0.2306 Accuracy: 93.41%
Epoch [7/10] Loss: 0.2236 Accuracy: 93.71%
Epoch [8/10] Loss: 0.2186 Accuracy: 93.92%
Epoch [9/10] Loss: 0.2074 Accuracy: 94.27%
Epoch [10/10] Loss: 0.2018 Accuracy: 94.48%
Test Accuracy: 82.64 %

Testing RMSprop: {'lr': 0.001, 'alpha': 0.99}
Epoch [1/10] Loss: 0.4226 Accuracy: 87.09%
Epoch [2/10] Loss: 0.2130 Accuracy: 93.62%
Epoch [3/10] Loss: 0.1505 Accuracy: 95.43%
Epoch [4/10] Loss: 0.1123 Accuracy: 96.66%
Epoch [5/10] Loss: 0.0861 Accuracy: 97.45%
Epoch [6/10] Loss: 0.0662 Accuracy: 97.95%
Epoch [7/10] Loss: 0.0524 Accuracy: 98.32%
Epoch [8/10] Loss: 0.0423 Accuracy: 98.69%
Epoch [9/10] Loss: 0.0342 Accuracy: 98.94%
Epoch [10/10] Loss: 0.0286 Accuracy: 99.08%
Test Accuracy: 90.12 %

In [15]:
results_df = pd.DataFrame(results)

results_df

,Optimizer,Learning Rate,Alpha,Test Accuracy,Training Time
0,RMSprop,0.005,0.9,87.97,98.815553


In [16]:
rms_results = results_df[
    results_df["Optimizer"] == "RMSprop"
]

best_rms = rms_results.sort_values(
    by="Test Accuracy",
    ascending=False
)

print(best_rms.head())

  Optimizer  Learning Rate  Alpha  Test Accuracy  Training Time
0   RMSprop          0.005    0.9          87.97      98.815553


In [17]:
best_rms_accuracy = best_rms.iloc[0]["Test Accuracy"]

print("Best RMSprop Accuracy:",
      round(best_rms_accuracy,2), "%")

Best RMSprop Accuracy: 87.97 %
